# Guide for Debugging Tutor Notebooks

### Using HuggingFace Hub + llama.cpp

This notebook downloads the three GGUF models used in the Debugging Tutor Demo and checks whether the environment is ready (CPU-only or ~4GB VRAM settings).

### What is the Debugging Tutor?
The demo is a AI assistant that helps students debug by guiding them through **Diagnosis → Root Cause → Check → Review**, instead of giving full solutions.

### Why are three small local models and a large API model used?
Three Qwen2.5 1.5B variants are used to demonstrate that model behavior depends not only on prompt and context, but also on training alignment. A large API model provides higher performance (optionally as an LLM-as-Judge), while highlighting cost and infrastructure tradeoffs. This comparison illustrates why model choice, evaluation, and training objectives matter in AI workflows.


## 0. Models
- Model Series: Qwen2.5 
- Parameters: 1.54B
- Q4 (4-bit) quantization for CPU-friendly
- 128K context window, 8K max generation.


| Model | Training Stage | Data | Notes |
| --- | --- | --- | --- |
| **Qwen2.5-1.5B** | Pretraining | General model trained on ~18T mixed tokens (text + some code) | Base language model |
| **Qwen2.5-Coder-1.5B** | Pretraining | Code-specialized model trained on ~5.5T code-focused tokens (source code, text-code grounding, synthetic data) | Significantly improvements in code generation, code reasoning and code fixing |
| **Qwen2.5-Coder-1.5B-Instruct** | Pretraining + Post-training | Coder + Instruction dataset | Coder + Significant improvements in instruction following, generating long texts, and generating structured outputs |


Model Cards: [Qwen2.5-1.5B](https://huggingface.co/QuantFactory/Qwen2.5-1.5B-GGUF), [Qwen2.5-Coder-1.5B](https://huggingface.co/QuantFactory/Qwen2.5-Coder-1.5B-GGUF), [Qwen2.5-Coder-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF).

References: `LlamaCpp_SmallLM_Demo.ipynb`, `HuggingFace_Hub_Download_gguf.ipynb`


## 1. Setup

Install and import HuggingFace Hub.


In [1]:
# Install/import HuggingFace Hub for model downloads.
try:
    from huggingface_hub import hf_hub_download, get_hf_file_metadata, hf_hub_url
except Exception:
    %pip install -q -U huggingface_hub
    from huggingface_hub import hf_hub_download, get_hf_file_metadata, hf_hub_url

from pathlib import Path
import os, time, shutil, subprocess


### 1.1 Pick your environment - Local vs Hub - and set the Path


In [2]:
# Hub path (DataHub / JupyterHub)
target_dir = Path('/home/jovyan/shared/')

# Local path example (if you run outside DataHub)
# target_dir = Path('./shared-rw').resolve()

target_dir.mkdir(parents=True, exist_ok=True)

# repo_id + filename: GGUF for llama.cpp & 4-bit quantized models (Q4) for CPU-only or ~4GB GPU.
# https://huggingface.co/repo_id/blob/main/filename
models = [
    {
        # https://huggingface.co/QuantFactory/Qwen2.5-1.5B-GGUF/blob/main/Qwen2.5-1.5B.Q4_K_M.gguf
        'repo_id': 'QuantFactory/Qwen2.5-1.5B-GGUF',
        'filename': 'Qwen2.5-1.5B.Q4_K_M.gguf',
    },
    {
        # https://huggingface.co/QuantFactory/Qwen2.5-Coder-1.5B-GGUF/blob/main/Qwen2.5-Coder-1.5B.Q4_K_M.gguf
        'repo_id': 'QuantFactory/Qwen2.5-Coder-1.5B-GGUF',
        'filename': 'Qwen2.5-Coder-1.5B.Q4_K_M.gguf',
    },
    {
        # https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/blob/main/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf
        'repo_id': 'Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF',
        'filename': 'qwen2.5-coder-1.5b-instruct-q4_k_m.gguf',
    },
]

print('target_dir:', target_dir)
for m in models:
    print(f"- {m['repo_id']} | {m['filename']}")


target_dir: /home/jovyan/shared
- QuantFactory/Qwen2.5-1.5B-GGUF | Qwen2.5-1.5B.Q4_K_M.gguf
- QuantFactory/Qwen2.5-Coder-1.5B-GGUF | Qwen2.5-Coder-1.5B.Q4_K_M.gguf
- Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF | qwen2.5-coder-1.5b-instruct-q4_k_m.gguf


## 2. Check Environment

Check model size, free disk space, and CPU-only or ~4GB GPU runtime settings.
Then copy the printed `use_gpu`, `n_ctx`, and `n_gpu_layers` into `Debugging_Tutor_Demo.ipynb`.



In [3]:
def detect_max_vram_gb():
    """Return max NVIDIA VRAM in GB, or None if unavailable."""
    try:
        out = subprocess.check_output(
            ['nvidia-smi', '--query-gpu=memory.total', '--format=csv,noheader,nounits'],
            stderr=subprocess.STDOUT,
            text=True,
        )
        values = [float(x.strip()) / 1024 for x in out.strip().splitlines() if x.strip()]
        return max(values) if values else None
    except Exception:
        return None


# 1) Check expected download size and free disk space.
rows = []
total_size_gb = 0.0
all_sizes_known = True

for m in models:
    try:
        url = hf_hub_url(repo_id=m['repo_id'], filename=m['filename'])
        meta = get_hf_file_metadata(url)
        size_gb = meta.size / (1024**3)
        rows.append((m['filename'], size_gb))
        total_size_gb += size_gb
    except Exception:
        rows.append((m['filename'], None))
        all_sizes_known = False

free_gb = shutil.disk_usage(str(target_dir)).free / (1024**3)

print('--- Download size check ---')
for name, size_gb in rows:
    if size_gb is None:
        print(f'{name}: size lookup failed')
    else:
        print(f'{name}: {size_gb:.2f} GB')

if all_sizes_known:
    print(f'Total expected download size: {total_size_gb:.2f} GB')
else:
    print('Total expected download size: unavailable')
print(f'Free disk in target path: {free_gb:.2f} GB')

if all_sizes_known:
    disk_ok = free_gb >= total_size_gb * 1.2
    print('Disk status:', 'PASS' if disk_ok else 'WARN (free more disk space)')
else:
    print('Disk status: CHECK MANUALLY')


# 2) Recommend simple CPU-only / ~4GB VRAM settings.
max_vram_gb = detect_max_vram_gb()
cpu_cores = os.cpu_count() or 0

# Start here for 1.5B Q4; lower if unstable(20 -> 16 -> 12 -> 8 / 2048 -> 1024).
recommended_n_ctx = 2048
recommended_use_gpu = bool(max_vram_gb is not None and max_vram_gb >= 4.0)
# 3B Q4 on ~4GB VRAM: start n_gpu_layers=8.
# 1B full/F16 on ~4GB VRAM: start n_gpu_layers=12~16.
recommended_n_gpu_layers = 20 if recommended_use_gpu else 0

print('\n--- Runtime suggestion ---')
print('CPU cores:', cpu_cores)
print('Max NVIDIA VRAM (GB):', f'{max_vram_gb:.2f}' if max_vram_gb is not None else 'not found')

if max_vram_gb is None:
    print('Mode: CPU-only')
elif max_vram_gb >= 4.0:
    print('Mode: ~4GB VRAM available')
else:
    print('Mode: low VRAM (start CPU-only)')

print('\nNext step: in Debugging_Tutor_Demo.ipynb, set:')
print('use_gpu =', recommended_use_gpu)
print('n_ctx =', recommended_n_ctx)
print('n_gpu_layers =', recommended_n_gpu_layers)
print('Start here for 1.5B Q4; if unstable, lower n_gpu_layers (20 -> 16 -> 12 -> 8).')



--- Download size check ---
Qwen2.5-1.5B.Q4_K_M.gguf: 0.92 GB
Qwen2.5-Coder-1.5B.Q4_K_M.gguf: 0.92 GB
qwen2.5-coder-1.5b-instruct-q4_k_m.gguf: 1.04 GB
Total expected download size: 2.88 GB
Free disk in target path: 5.00 GB
Disk status: PASS

--- Runtime suggestion ---
CPU cores: 64
Max NVIDIA VRAM (GB): 22.49
Mode: ~4GB VRAM available

Next step: in Debugging_Tutor_Demo.ipynb, set:
use_gpu = True
n_ctx = 2048
n_gpu_layers = 20
Start here for 1.5B Q4; if unstable, lower n_gpu_layers (20 -> 16 -> 12 -> 8).


## 3. Download the model files

Each file is downloaded with retry logic for temporary network failures.


In [4]:
# Download each model with retry for temporary network errors.
def download_one(repo_id: str, filename: str, retries: int = 3):
    for attempt in range(1, retries + 1):
        try:
            return hf_hub_download(
                repo_id=repo_id,
                filename=filename,
                local_dir=str(target_dir),
            )
        except Exception as e:
            print(f'Attempt {attempt}/{retries} failed for {filename}: {e}')
            if attempt == retries:
                raise
            time.sleep(2 ** attempt)

for m in models:
    print()
    print('Downloading:', m['filename'])
    saved = download_one(m['repo_id'], m['filename'])
    print('Saved:', saved)



Downloading: Qwen2.5-1.5B.Q4_K_M.gguf
Saved: /home/jovyan/shared/Qwen2.5-1.5B.Q4_K_M.gguf

Downloading: Qwen2.5-Coder-1.5B.Q4_K_M.gguf
Saved: /home/jovyan/shared/Qwen2.5-Coder-1.5B.Q4_K_M.gguf

Downloading: qwen2.5-coder-1.5b-instruct-q4_k_m.gguf
Saved: /home/jovyan/shared/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf


## 4. Verify downloaded files

Confirm that all target files exist in the selected directory.


In [5]:
# Verify each downloaded file exists and print its size.
missing = []
for m in models:
    f = target_dir / m['filename']
    if not f.exists():
        missing.append(m['filename'])
    else:
        print(f'{f.name}: {f.stat().st_size / (1024**3):.2f} GB')

print('Missing files:', missing)
assert len(missing) == 0, 'Some model files are missing.'
print()
print('PASS: all listed model files are downloaded.')


Qwen2.5-1.5B.Q4_K_M.gguf: 0.92 GB
Qwen2.5-Coder-1.5B.Q4_K_M.gguf: 0.92 GB
qwen2.5-coder-1.5b-instruct-q4_k_m.gguf: 1.04 GB
Missing files: []

PASS: all listed model files are downloaded.


## Next Steps

- **Intro**: Demo + experiments
- **Intermediate**: Routing + RAG + metadata pipeline automation (optional metrics)
- **Advanced**: Multi-Agent System (RAG agent + Exam agent + Writing agent + Debugging agent)
- **Advanced(optional, GPU)**: Synthetic data pipeline + LoRA/distillation 
